# Load data

In [1]:
# Load the data and rebuild the fair, de-noised train / val sets
# Why: same data as the 0.92 baseline, so the Transformer can be compared with it directly.
import json, re
from pathlib import Path
import pandas as pd

rows = [json.loads(l) for l in open("data/raw/dataset_clean.jsonl", encoding="utf-8") if l.strip()]
df = pd.DataFrame(rows).drop_duplicates(subset=["title_ar", "description_ar"]).reset_index(drop=True)

NOISY = ["TCKT-008", "TCKT-011", "TCKT-013", "TCKT-014"]       # batches with scrambled labels (Finding 16)
df = df[~df["ticket_id"].str[:8].isin(NOISY)]

LEAK = re.compile(r"فيه مشكلة مرتبطة بـ[^.]*\.")              # the leak sentence (Finding 9)
df["text"] = df["title_ar"] + " . " + df["description_ar"].str.replace(LEAK, "", regex=True)

def load_ids(name):
    return set(Path(f"data/splits/{name}_ids.txt").read_text(encoding="utf-8").split())

train = df[df["ticket_id"].isin(load_ids("train"))].reset_index(drop=True)
val   = df[df["ticket_id"].isin(load_ids("val"))].reset_index(drop=True)
print(len(train), len(val))          # expected: 5269 1251  (test stays closed)

5269 1251


In [2]:
# Normalize the text and split it into word tokens
# Why: "الإيميل" and "الايميل" (or "VPN" and "vpn") should be the same token for the model.
def normalize(text):
    text = re.sub("[إأآ]", "ا", text)              # unify the alef forms
    text = text.replace("ى", "ي")                  # alef maqsura -> ya
    text = re.sub("[\u064B-\u0652\u0640]", "", text)  # remove diacritics (tashkeel) and tatweel "ـ"
    return text.lower()                            # English letters to lowercase

def tokenize(text):
    return re.findall(r"\w+", normalize(text))     # a token = a run of letters or digits

print(tokenize(train["text"][0]))

['السيستم', 'واقع', 'يا', 'جماعة', 'السيستم', 'كله', 'مش', 'شغال', 'عند', 'كل', 'الناس', 'في', 'القسم', 'في', 'عطل', 'عام', 'بقالها', '21', 'ساعة', 'تقريبا', 'لو', 'محتاجين', 'لوجز', 'او', 'سكرينشوت', 'انا', 'جاهز', 'ابعتها']


In [3]:
# Build the vocabulary from TRAIN only: every word seen at least twice gets an id
# Why: the model can only look up words that have an id; val/test words must never shape it.
from collections import Counter

counts = Counter(tok for text in train["text"] for tok in tokenize(text))
SPECIALS = ["[PAD]", "[UNK]", "[CLS]"]                          # ids 0, 1, 2
vocab = SPECIALS + [w for w, c in counts.most_common() if c >= 2]
word2id = {w: i for i, w in enumerate(vocab)}                   # word -> id
print("vocabulary size:", len(vocab))
print(vocab[:15])                                               # specials + the most frequent words

vocabulary size: 1422
['[PAD]', '[UNK]', '[CLS]', 'علي', 'المشكلة', 'ساعة', 'بقالها', 'تقريبا', 'في', 'لو', 'جربت', 'نفس', 'ومش', 'الموضوع', 'مش']


In [4]:
# Turn a text into ids: [CLS] first, then one id per word (unknown words -> [UNK])
# Why: the model only understands numbers; [CLS] is the "summary seat" at position 0.
PAD, UNK, CLS = word2id["[PAD]"], word2id["[UNK]"], word2id["[CLS]"]

def encode(text):
    return [CLS] + [word2id.get(tok, UNK) for tok in tokenize(text)]

print(train["text"][0])
print(encode(train["text"][0]))

السيستم واقع . يا جماعة السيستم كله مش شغال عند كل الناس في القسم، في عطل عام؟. بقالها 21 ساعة تقريبا. لو محتاجين لوجز أو سكرينشوت أنا جاهز أبعتها.
[2, 99, 274, 62, 85, 99, 370, 14, 67, 413, 111, 560, 8, 602, 8, 116, 283, 6, 149, 5, 7, 9, 22, 23, 43, 51, 20, 24, 52]


In [5]:
# How long are the tickets (in tokens)? And how many val words are unknown?
# Why: we must pick ONE length for all tickets; many unknown words would mean the vocabulary is too small.
train_lengths = train["text"].map(lambda t: len(encode(t)))
print(train_lengths.describe().round(1))
print("95% of tickets have at most", int(train_lengths.quantile(0.95)), "tokens")

val_tokens = [tok for t in val["text"] for tok in tokenize(t)]
unknown = sum(tok not in word2id for tok in val_tokens)
print("unknown words in val:", round(unknown / len(val_tokens) * 100, 1), "%")

count    5269.0
mean       30.9
std         8.4
min         9.0
25%        26.0
50%        29.0
75%        34.0
max        67.0
Name: text, dtype: float64
95% of tickets have at most 49 tokens
unknown words in val: 2.1 %


In [6]:
# Make every ticket exactly MAX_LEN ids long: cut long ones, fill short ones with [PAD]
# Why: the model processes a batch as one rectangle (tickets × positions), so all rows need the same length.
MAX_LEN = 64

def encode_fixed(text):
    ids = encode(text)[:MAX_LEN]                  # cut if too long
    return ids + [PAD] * (MAX_LEN - len(ids))     # fill with [PAD] (= 0) if too short

print("tickets longer than MAX_LEN:", (train_lengths > MAX_LEN).sum())
example = encode_fixed(train["text"][0])
print(len(example), example)

tickets longer than MAX_LEN: 10
64 [2, 99, 274, 62, 85, 99, 370, 14, 67, 413, 111, 560, 8, 602, 8, 116, 283, 6, 149, 5, 7, 9, 22, 23, 43, 51, 20, 24, 52, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [7]:
# Encode all tickets into one matrix of ids, and the labels into class numbers
# Why: this matrix (tickets × MAX_LEN) is exactly what the Transformer will receive as input.
import numpy as np

X_train_ids = np.array([encode_fixed(t) for t in train["text"]])
X_val_ids   = np.array([encode_fixed(t) for t in val["text"]])

classes = sorted(train["category_level_2"].unique())       # the 16 L2 names, in a fixed order
class2id = {c: i for i, c in enumerate(classes)}           # "VPN" -> a number from 0 to 15
y_train_ids = train["category_level_2"].map(class2id).to_numpy()
y_val_ids   = val["category_level_2"].map(class2id).to_numpy()

print(X_train_ids.shape, X_val_ids.shape)                  # expected: (5269, 64) (1251, 64)
print(classes)

(5269, 64) (1251, 64)
['Account', 'Business App', 'Email/Calendar', 'Incident', 'Internet/LAN', 'Laptop/Desktop', 'MFA/SSO', 'Malware/Phishing', 'Office Apps', 'Peripherals', 'Permissions', 'Policy/Compliance', 'Printer/Scanner', 'Request', 'VPN', 'WiFi']


In [8]:
# See what the random generator produces, and that the seed makes it repeatable
rng = np.random.default_rng(0)
print(rng.normal(0, 1, size=5))     # 5 random numbers around 0

rng = np.random.default_rng(0)
print(rng.normal(0, 1, size=5))     # same seed -> exactly the same 5 numbers

[ 0.12573022 -0.13210486  0.64042265  0.10490012 -0.53566937]
[ 0.12573022 -0.13210486  0.64042265  0.10490012 -0.53566937]


In [9]:
# Embedding by hand: a table E with one row (vector) per word id
# Why: an id like 5 is only a name; the model needs a vector of numbers it can learn and compute with.
rng = np.random.default_rng(0)
d_model = 8                                         # vector size per token (tiny, so we can look at it)
E = rng.normal(0, 1, size=(len(vocab), d_model))    # shape (vocab size, d_model), random for now
print("E shape:", E.shape)

ids = X_train_ids[:2]                               # a mini-batch: 2 tickets × 64 positions
vectors = E[ids]                                    # look up one row of E for every id
print("ids shape:", ids.shape, "-> vectors shape:", vectors.shape)

E shape: (1422, 8)
ids shape: (2, 64) -> vectors shape: (2, 64, 8)


In [10]:
# The same lookup written as a matrix multiplication: one_hot(id) @ E picks exactly row "id"
# Why: shows that an embedding is just a linear layer applied to a one-hot vector.
word_id = ids[0, 1]                                 # first real word of ticket 0 (position 0 is [CLS])
one_hot = np.zeros(len(vocab))
one_hot[word_id] = 1                                # all zeros except a single 1 at position word_id
print(np.allclose(one_hot @ E, E[word_id]))         # expected: True

True


In [11]:
# Every [PAD] position gets the same vector: row 0 of E
# Why: padding carries no meaning; later the attention mask will tell the model to ignore it.
print(np.allclose(vectors[0, -1], E[PAD]))          # last position of ticket 0 is padding -> True

True


In [12]:
# Positional embedding by hand: a second table P with one row per POSITION (0 .. MAX_LEN-1)
# Why: attention alone does not know word order; adding P makes the same word look different at each position.
P = rng.normal(0, 1, size=(MAX_LEN, d_model))      # shape (64, 8): one vector per position
print("P shape:", P.shape)

H0 = E[ids] + P                                     # (2, 64, 8) + (64, 8) -> (2, 64, 8)
print("H0 shape:", H0.shape)

P shape: (64, 8)
H0 shape: (2, 64, 8)


In [13]:
# The same word at two different positions now gets two different vectors
# Why: this is how the model can tell "بيوصل بس مش بيبعت" from "بيبعت بس مش بيوصل".
w = word2id["مش"]
print("without P, position 3 vs 7 equal?", np.allclose(E[w], E[w]))                # True: the word alone has no position
print("with P,    position 3 vs 7 equal?", np.allclose(E[w] + P[3], E[w] + P[7]))  # expected: False

without P, position 3 vs 7 equal? True
with P,    position 3 vs 7 equal? False


In [14]:
# Q, K, V: three different "views" of every token, made by three weight matrices
# Why: each token needs a question (Q), a label to be found by (K), and the information it shares (V).
W_Q = rng.normal(0, 1, size=(d_model, d_model)) / np.sqrt(d_model)   # (8, 8); random now, learned in training
W_K = rng.normal(0, 1, size=(d_model, d_model)) / np.sqrt(d_model)
W_V = rng.normal(0, 1, size=(d_model, d_model)) / np.sqrt(d_model)

Q = H0 @ W_Q          # (2, 64, 8) @ (8, 8) -> (2, 64, 8)
K = H0 @ W_K
V = H0 @ W_V
print("Q", Q.shape, "| K", K.shape, "| V", V.shape)

Q (2, 64, 8) | K (2, 64, 8) | V (2, 64, 8)


In [15]:
# Attention scores: how much each token (row i) looks at each other token (column j)
# Why: Q·K is large when a question matches a label; dividing by sqrt(d) keeps the numbers moderate.
scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_model)   # (2, 64, 8) @ (2, 8, 64) -> (2, 64, 64)
print("scores shape:", scores.shape)

print([vocab[i] for i in ids[0, :5]])                  # the first 5 tokens of ticket 0
print(scores[0, :5, :5].round(2))                      # how these 5 tokens look at each other

scores shape: (2, 64, 64)
['[CLS]', 'السيستم', 'واقع', 'يا', 'جماعة']
[[ -0.93  -0.99   1.42  -0.03  -1.2 ]
 [ -0.02  -2.81   2.38  -0.96  -0.44]
 [ -0.68   4.87  -4.05   1.5   -1.35]
 [  2.1   -4.19   3.29  -2.28   4.41]
 [ -5.33   4.43  -1.72   2.97 -10.24]]
